# 🚀 Welcome to Your Redis Semantic Caching Sandbox

You're about to build an intelligent semantic caching system using Redis. The catch? It's open-ended—choose a domain (e-commerce support, travel recommendations, code documentation, or invent your own) and create a caching strategy that understands semantic similarity.

**Your first step:** Choose your domain and write a specification defining your caching strategy, similarity thresholds, and cache invalidation rules.

> 💡 **New to JupyterAI?** Learn more about coding with AI in notebooks at [JupyterAI: Coding in Notebooks](https://www.deeplearning.ai/short-courses/jupyter-ai-coding-in-notebooks/)

> ⚠️ **Important:** Your workspace session lasts 2 hours. Remember to download `project.ipynb` and `spec.md` regularly to save your progress!

## What you'll craft step by step
1. **Spec File** 📝 — Define your caching strategy and similarity parameters
2. **Redis Setup** 🔧 — Configure Redis vector store with embeddings
3. **Embedding Model** 🧠 — Select and implement semantic embeddings
4. **Build Cache** 💾 — Implement core semantic caching logic
5. **Cache Retrieval** 🔍 — Query similarity and threshold logic
6. **Test & Evaluate** 📊 — Measure performance and hit rates

```
      What you'll craft step by step

              [ Spec File 📝 ]
                      |
                      v
            [ Redis Setup 🔧 ]
                      |
                      v
           [ Embedding Model 🧠 ]
                      |
                      v
             [ Build Cache 💾 ]
                      |
                      v
           [ Cache Retrieval 🔍 ]
                      |
                      v
           [ Test & Evaluate 📊 ]
```

🔴 Need guidance? Use the left-hand chatbot for targeted questions at any stage.

* To open Jupyter chat, click on the chat bubble icon on the left sidebar of Jupyter Lab:
  
  <img src="images/jupyter_chat_bordered.png" width="150" style="vertical-align: middle;">

Pick a use case, define your caching strategy, and let's build an intelligent cache.

<details>
<summary><strong>🗂️ Reminder: Attach context to every chatbot prompt</strong></summary>

When you ask the chatbot for help, always upload these files so it has full project context:

- `project.ipynb`: shares your latest code and notebook state
- `spec.md`: your semantic caching system specification
- Redis documentation (provided in resources)

Bringing all context keeps responses grounded in what you've built, Redis capabilities, and your caching design.

**Debugging tip:** When troubleshooting issues, share error messages and relevant code snippets with the chatbot. Follow iterative debugging practices—test small changes, verify outputs, and use the AI assistant to help diagnose problems step by step.
</details>

## Setup and Imports

*Optional reading. Skip if you are familiar with import statements.*

First, we'll import all the necessary libraries and set up our environment. This project uses:
- **Redis** - For vector storage and semantic search
- **RedisVL** - Redis Vector Library for semantic caching
- **Sentence Transformers** - For generating semantic embeddings
- **Python-dotenv** - For environment variable management
- **cache module** - Helper utilities for cache wrapper and evaluation (provided)

### Helper Utilities

This project includes helper utilities in the `cache/` folder:
- **`cache.config`** - Configuration and API key loading
- **`cache.wrapper`** - `SemanticCacheWrapper` class for simplified cache operations
- **`cache.evals`** - `CacheEvaluator` and `PerfEval` for measuring performance

These helpers follow the same patterns used in the course lessons, making it easier to build and evaluate your semantic cache.

### Useful Resources:
- **Redis Vector Search**: [https://redis.io/docs/latest/develop/ai/search-and-query/vectors/](https://redis.io/docs/latest/develop/ai/search-and-query/vectors/)
- **RedisVL Documentation**: [https://docs.redisvl.com/](https://docs.redisvl.com/)
- **Semantic Caching Guide**: [https://redis.io/blog/what-is-semantic-caching/](https://redis.io/blog/what-is-semantic-caching/)
- **Sentence Transformers**: [https://www.sbert.net/](https://www.sbert.net/)

### Important Note
Semantic caching can reduce **LLM API costs by 70%+** and **latency by 80%+** by intelligently reusing responses to semantically similar queries.

### Note: You should also consult the chatbot on the left to ask more questions about this project or about Redis vector search.

In [ ]:
import os
import json
import warnings
from typing import Dict, List, Optional, Any
from datetime import datetime

# Suppress warnings
warnings.filterwarnings('ignore')

# Redis and RedisVL imports
import redis
from redisvl.extensions.llmcache import SemanticCache
from sentence_transformers import SentenceTransformer

# Import helper utilities
from cache.config import config, load_openai_key
from cache.wrapper import SemanticCacheWrapper, try_connect_to_redis
from cache.evals import CacheEvaluator, PerfEval

In [ ]:
# Configure environment using the cache config
from cache.config import config

REDIS_URL = config.get("redis_url", "redis://localhost:6379")

try:
    client = redis.from_url(REDIS_URL)
    client.ping()
    print("Redis connection successful")
    print(f"Connected to: {REDIS_URL}")
except Exception as e:
    print("Redis connection failed!")
    print(f"Error: {e}")
    print("Please start Redis with: docker run -d --name redis -p 6379:6379 redis/redis-stack:latest")

# 📝 Spec File

A spec file captures your entire semantic caching strategy. Draft it _before_ writing code so you and your AI helpers stay aligned on cache design, similarity thresholds, and invalidation rules.

You need to write it using the chatbot and create a file called `spec.md` that you will be attaching to all further interactions with the chatbot.

- ✅ **Purpose:** Define your caching ecosystem—what to cache, similarity thresholds, TTL policies, and cache invalidation strategies.
- 🧠 **Brainstorm:** Think of use cases with repeated similar queries (customer support FAQs, travel recommendations, code questions).
- 🧩 **Focus:** Define distance metrics, embedding models, and threshold tuning strategies.

> **Recommendation:** Start with a similarity threshold around 0.85-0.90 for cosine distance. You can tune this based on hit rate vs. accuracy trade-offs.

---

## ✍️ What your `spec.md` should include
- **System Name** — brief label (e.g., "E-commerce Support Cache").
- **Purpose** — one sentence describing what queries you're caching.
- **Domain** — describe your use case and typical query patterns.
- **Embedding Strategy:**
  - Model choice (e.g., all-MiniLM-L6-v2, multilingual models)
  - Embedding dimension
  - Why this model fits your domain
- **Cache Configuration:**
  - Similarity threshold (e.g., 0.85 cosine similarity)
  - Distance metric (cosine, euclidean, inner product)
  - TTL (Time-To-Live) policy
  - Index type (FLAT, HNSW)
- **Cache Strategy:**
  - What triggers a cache write
  - How to handle cache misses
  - Cache invalidation rules
- **Performance Goals:**
  - Target hit rate (e.g., >60%)
  - Acceptable latency (e.g., <5ms for cache lookup)
  - Cost reduction target (e.g., 70% fewer LLM API calls)
- **Example Queries:**
  - Sample queries that should hit cache
  - Sample queries that should miss cache

---

<details>
<summary><strong>📎 Example Spec — E-commerce Customer Support Cache (click to expand)</strong></summary>

```
# E-commerce Customer Support Semantic Cache

**System Name:** E-commerce Support Cache

**Purpose:** Cache responses to customer support questions to reduce LLM API costs and improve response times.

**Domain:** E-commerce customer support handling refund requests, shipping inquiries, product questions, and account issues.

## Embedding Strategy

- **Model:** `all-MiniLM-L6-v2` (sentence-transformers)
- **Dimension:** 384
- **Rationale:** Fast, efficient, good performance on short queries typical in customer support

## Cache Configuration

- **Similarity Threshold:** 0.87 (cosine distance)
- **Distance Metric:** Cosine similarity
- **TTL:** 7 days (balance freshness with hit rate)
- **Index Type:** HNSW (faster retrieval for large caches)

## Cache Strategy

**Cache Write Triggers:**
- After successfully generating LLM response
- Store: {query_embedding, response_text, metadata, timestamp}

**Cache Miss Handling:**
1. Check semantic cache with threshold
2. If miss → call LLM API
3. Store new response in cache

**Cache Invalidation:**
- TTL-based expiration (7 days)
- Manual flush for policy changes
- Product catalog updates trigger related cache clears

## Performance Goals

- **Target Hit Rate:** 65%+
- **Cache Lookup Latency:** <3ms
- **Cost Reduction:** 70% fewer LLM API calls
- **Precision:** >95% (cached responses match user intent)

## Example Queries

**Should Hit Cache (similar meaning):**
- "How do I get a refund?" ↔ "What's your refund policy?"
- "Where is my order?" ↔ "How can I track my shipment?"
- "Cancel my subscription" ↔ "I want to stop my recurring payment"

**Should Miss Cache (different intent):**
- "How do I get a refund?" ✗ "What payment methods do you accept?"
- "Where is my order?" ✗ "Do you ship internationally?"
```

</details>

<details>
<summary><strong>🌍 Example Spec — Travel Recommendation Cache (click to expand)</strong></summary>

```
# Travel Recommendation Semantic Cache

**System Name:** Travel Recommendation Cache

**Purpose:** Cache responses to travel destination queries to reduce API costs and provide faster recommendations.

**Domain:** Travel booking assistant handling destination recommendations, vacation planning, and travel advice.

## Embedding Strategy

- **Model:** `paraphrase-multilingual-MiniLM-L12-v2` (sentence-transformers)
- **Dimension:** 384
- **Rationale:** Multilingual support for international destinations, good performance on travel queries

## Cache Configuration

- **Similarity Threshold:** 0.85 (cosine distance)
- **Distance Metric:** Cosine similarity
- **TTL:** 30 days (seasonal consistency for travel recommendations)
- **Index Type:** HNSW (faster retrieval for large caches)

## Cache Strategy

**Cache Write Triggers:**
- After successfully generating travel recommendation
- Store: {query_embedding, response_text, destination_tags, timestamp}

**Cache Miss Handling:**
1. Check semantic cache with threshold
2. If miss → call LLM API
3. Store new response in cache

**Cache Invalidation:**
- TTL-based expiration (30 days)
- Seasonal updates (summer/winter destinations)
- Price/availability changes trigger related cache clears

## Performance Goals

- **Target Hit Rate:** 60%+
- **Cache Lookup Latency:** <5ms
- **Cost Reduction:** 65% fewer LLM API calls
- **Precision:** >90% (recommendations match user intent)

## Example Queries

**Should Hit Cache (similar meaning):**
- "Best beaches in Europe" ↔ "Top European beach destinations"
- "Affordable ski resorts" ↔ "Budget skiing locations"
- "Family-friendly cities" ↔ "Best cities for kids"

**Should Miss Cache (different intent):**
- "Best beaches in Europe" ✗ "Best hiking trails in Europe"
- "Affordable ski resorts" ✗ "Luxury ski resorts"
```

</details>

<details>
<summary><strong>💻 Example Spec — Code Documentation Cache (click to expand)</strong></summary>

```
# Code Documentation Semantic Cache

**System Name:** Code Documentation Cache

**Purpose:** Cache answers to programming questions to reduce API costs and provide instant coding help.

**Domain:** Developer assistant handling syntax questions, code examples, and programming explanations.

## Embedding Strategy

- **Model:** `multi-qa-MiniLM-L6-cos-v1` (sentence-transformers)
- **Dimension:** 384
- **Rationale:** Optimized for Q&A, good performance on technical content

## Cache Configuration

- **Similarity Threshold:** 0.88 (cosine distance)
- **Distance Metric:** Cosine similarity
- **TTL:** 14 days (balance freshness with hit rate)
- **Index Type:** HNSW (faster retrieval for large caches)

## Cache Strategy

**Cache Write Triggers:**
- After successfully generating code explanation
- Store: {query_embedding, response_text, language_tag, timestamp}

**Cache Miss Handling:**
1. Check semantic cache with threshold
2. If miss → call LLM API
3. Store new response in cache

**Cache Invalidation:**
- TTL-based expiration (14 days)
- Framework/language version updates
- Deprecated syntax triggers cache clear

## Performance Goals

- **Target Hit Rate:** 70%+
- **Cache Lookup Latency:** <3ms
- **Cost Reduction:** 75% fewer LLM API calls
- **Precision:** >95% (code answers must be accurate)

## Example Queries

**Should Hit Cache (similar meaning):**
- "Sort array Python" ↔ "How to sort list Python"
- "JavaScript async await" ↔ "JS async await syntax"
- "Python loop through dictionary" ↔ "Iterate dict Python"

**Should Miss Cache (different intent):**
- "Sort array Python" ✗ "Sort array JavaScript"
- "JavaScript async await" ✗ "JavaScript promises"
```

</details>

Keep your spec concise—something you can reference quickly when implementing cache logic and tuning thresholds.

In [ ]:
# Create your spec.md file using the chatbot
# This file will be attached to all future prompts

# Redis Setup & Vector Index

Time to configure Redis for semantic search. Redis needs a vector index to perform efficient similarity searches.

- **Goal:** Create a Redis vector index with the right configuration for your use case.
- **Workflow:** Reference your `spec.md` for embedding dimensions, distance metric, and index type.
- **Remember:** HNSW indexes are faster for large datasets, FLAT indexes are simpler for small caches.

---

## Redis Vector Index Configuration

Key parameters to configure:

1. **Index Name** - Unique identifier for your cache index
2. **Vector Dimension** - Must match your embedding model (e.g., 384 for MiniLM)
3. **Distance Metric** - COSINE (most common), L2 (Euclidean), or IP (inner product)
4. **Index Type** - FLAT (exact search) or HNSW (approximate, faster)
5. **TTL** - How long to keep cached entries

---

## Using the SemanticCacheWrapper Helper

The `cache.wrapper` module provides a `SemanticCacheWrapper` class that simplifies cache operations:

```python
from cache.wrapper import SemanticCacheWrapper
from cache.config import config

# Option 1: Create from config
cache = SemanticCacheWrapper.from_config(config)

# Option 2: Create with custom parameters
cache = SemanticCacheWrapper(
    name="my_cache",
    distance_threshold=0.3,
    ttl=3600,
    redis_url="redis://localhost:6379"
)

# Hydrate cache from DataFrame
cache.hydrate_from_df(df, q_col="question", a_col="answer")

# Or from list of pairs
cache.hydrate_from_pairs([("question1", "answer1"), ("question2", "answer2")])

# Check cache
results = cache.check("user query")
if results.matches:
    print(f"Cache hit: {results.matches[0].response}")

# Store new entry
cache.store(prompt="new question", response="new answer")
```

**Important notes:**
- Distance threshold works inverse to similarity: lower distance = higher similarity
- For cosine distance: 0.0 = identical, 2.0 = opposite
- Threshold of 0.1-0.15 corresponds to ~85-90% similarity

**Pro tip:** Start with a conservative threshold (0.1-0.15) and tune based on hit rate vs. precision metrics.

---

<details>
<summary><strong>Prompt Example - E-commerce Cache Setup (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to set up Redis semantic cache for e-commerce support.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Use SemanticCacheWrapper.from_config() or create with custom parameters
2. Create cache instance named "cache"
3. Set distance_threshold to 0.13 (approximately 87% cosine similarity)
4. Set TTL to 604800 seconds (7 days)
5. Print confirmation message when cache is initialized
6. Handle connection errors gracefully
7. Hydrate cache with sample FAQ data using cache.hydrate_from_df()

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>Prompt Example - Travel Cache Setup (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to set up Redis semantic cache for travel recommendations.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Use SemanticCacheWrapper with custom parameters
2. Create cache instance named "cache"
3. Set distance_threshold to 0.15 (looser matching for travel queries)
4. Set TTL to 2592000 seconds (30 days for seasonal consistency)
5. Print cache statistics after initialization
6. Hydrate cache with sample travel data using cache.hydrate_from_df()

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>Prompt Example - Code Documentation Cache Setup (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to set up Redis semantic cache for code documentation.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Use SemanticCacheWrapper.from_config() with config overrides
2. Create cache instance named "cache"
3. Set distance_threshold to 0.12 (stricter for code accuracy)
4. Set TTL to 1209600 seconds (14 days)
5. Include cache size monitoring
6. Hydrate cache with sample coding Q&A data using cache.hydrate_from_df()

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

In [ ]:
# Empty cell for student to paste Redis cache setup code

# 🧠 Embedding Model Selection

> ⏰ **Reminder:** Don't forget to save your work! Download `project.ipynb` and `spec.md` to preserve your progress.

Choose and load the embedding model that will convert text queries into vectors for semantic comparison.

- 🎯 **Goal:** Initialize an embedding model suited to your domain.
- 🔁 **Workflow:** Reference your `spec.md` for the chosen model and rationale.
- 💡 **Remember:** Model choice impacts cache performance—smaller models are faster but may miss semantic nuances.

---

## 🔍 Popular Embedding Models

| Model | Dimension | Speed | Use Case |
|-------|-----------|-------|----------|
| `all-MiniLM-L6-v2` | 384 | Fast | General purpose, short text |
| `all-mpnet-base-v2` | 768 | Medium | Higher quality, longer text |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | Fast | Multilingual support |
| `multi-qa-MiniLM-L6-cos-v1` | 384 | Fast | Question-answering optimized |

---

## 🧱 Loading Sentence Transformers

```python
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embedding
query = "How do I get a refund?"
embedding = model.encode(query)
print(f"Embedding dimension: {len(embedding)}")
```

📌 **Pro tip:** Test multiple models on sample queries from your domain to find the best precision/speed trade-off.

---

<details>
<summary><strong>🛒 Prompt Example — E-commerce Embedding Model (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to initialize the embedding model for e-commerce support cache.
Return code only—no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Import SentenceTransformer from sentence_transformers
2. Load model: all-MiniLM-L6-v2 (384 dimensions)
3. Create a helper function encode_query(text: str) that returns embedding
4. Test with sample query: "How do I return an item?"
5. Print embedding shape and first 5 values
6. Verify dimension matches spec (384)

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>🌍 Prompt Example — Travel Embedding Model (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to initialize the embedding model for travel recommendations.
Return code only—no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Import SentenceTransformer
2. Load model: paraphrase-multilingual-MiniLM-L12-v2 (for multilingual travel queries)
3. Create encode_query(text: str) helper function
4. Test with: "Best beaches in Europe for families"
5. Print embedding statistics (dimension, mean, std)
6. Include batch encoding function for multiple queries

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>💻 Prompt Example — Code Documentation Embedding Model (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to initialize the embedding model for code documentation.
Return code only—no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Import SentenceTransformer
2. Load model: multi-qa-MiniLM-L6-cos-v1 (optimized for Q&A)
3. Create encode_query(text: str) function
4. Test with: "How to sort a list in Python?"
5. Compare embeddings of similar code questions
6. Print cosine similarity between test queries

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

In [ ]:
# Empty cell for student to paste embedding model code

# Build Semantic Cache Logic

Implement the core caching functions: storing responses and checking for semantic matches.

- **Goal:** Create functions to store and retrieve cached responses based on semantic similarity.
- **Workflow:** Use the SemanticCacheWrapper helper or implement custom logic.
- **Remember:** Cache writes should be fast-don't block on LLM responses.

---

## Core Cache Operations with SemanticCacheWrapper

The `SemanticCacheWrapper` provides convenient methods for cache operations:

**Store in cache:**
```python
# Single entry
cache.store(prompt="User query text", response="LLM response text")

# Bulk load from DataFrame
cache.hydrate_from_df(df, q_col="question", a_col="answer")

# Bulk load from pairs
cache.hydrate_from_pairs([("q1", "a1"), ("q2", "a2")])
```

**Check cache:**
```python
results = cache.check("Similar user query")

if results.matches:
    # Cache hit!
    cached_response = results.matches[0].response
    distance = results.matches[0].vector_distance
    similarity = results.matches[0].cosine_similarity
else:
    # Cache miss - call LLM
    pass
```

---

## Cache Wrapper Pattern

```python
def get_cached_or_generate(query: str, llm_function) -> str:
    """Check cache first, generate if miss."""
    # Check cache
    results = cache.check(query)
    
    if results.matches:
        print(f"Cache HIT for: {query[:50]}...")
        return results.matches[0].response
    
    # Cache miss - generate new response
    print(f"Cache MISS for: {query[:50]}...")
    response = llm_function(query)
    
    # Store in cache
    cache.store(prompt=query, response=response)
    
    return response
```

**Pro tip:** Add logging to track hit rates and monitor cache performance over time.

---

<details>
<summary><strong>Prompt Example - E-commerce Cache Functions (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code for semantic cache storage and retrieval functions.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Cache instance already initialized as: cache (SemanticCacheWrapper)

Requirements:
1. Create get_cached_or_generate(query: str, generate_fn) wrapper function
2. Use cache.check() and cache.store() methods
3. Add hit/miss tracking with counters
4. Include print statements showing cache status
5. Test with sample support queries

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>Prompt Example - Travel Cache Functions (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code for travel recommendation cache functions.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Cache instance: cache (SemanticCacheWrapper)

Requirements:
1. Create cache_or_search(query: str, search_fn) wrapper
2. Use cache.check() to get CacheResults
3. Access results.matches[0].response for cached data
4. Include cache statistics tracking (hits, misses, hit_rate)
5. Test with travel destination queries

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>Prompt Example - Code Documentation Cache Functions (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code for code documentation cache functions.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Cache instance: cache (SemanticCacheWrapper)

Requirements:
1. Create cache_or_lookup(query: str, lookup_fn) wrapper
2. Use cache.check() and cache.store()
3. Track language-specific cache statistics
4. Add programming language tags to stored metadata
5. Test with coding questions

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

In [ ]:
# Empty cell for student to paste cache logic code

# 🔍 Cache Retrieval & Threshold Tuning

Optimize cache retrieval by tuning similarity thresholds and implementing smart query strategies.

- 🎯 **Goal:** Find the optimal balance between hit rate and precision.
- 🔁 **Workflow:** Test different thresholds and measure impact on cache performance.
- 💡 **Remember:** Lower threshold = stricter matching = fewer hits but higher precision.

---

## 🔍 Threshold Tuning Strategy

**Trade-off analysis:**
- **Threshold 0.05-0.10**: Very strict, high precision, low hit rate
- **Threshold 0.10-0.15**: Balanced, good for production (85-90% similarity)
- **Threshold 0.15-0.25**: Loose, high hit rate, risk of false positives

> ⚠️ **Caution:** If you're getting zero cache hits, your threshold may be too strict. Experiment with higher threshold values (e.g., 0.2-0.3) and gradually tune down for better precision.

**Testing approach:**
```python
thresholds = [0.05, 0.10, 0.15, 0.20]
test_queries = [
    ("query1", "expected_similar"),
    ("query2", "expected_different")
]

for threshold in thresholds:
    cache.set_threshold(threshold)
    hit_rate, precision = evaluate_cache(test_queries)
    print(f"Threshold {threshold}: Hit={hit_rate:.2%}, Precision={precision:.2%}")
```

---

## 🧱 Query Preprocessing

Improve cache hits with query normalization:

```python
def normalize_query(query: str) -> str:
    """Normalize query for better cache matching."""
    # Lowercase
    query = query.lower()
    # Remove extra whitespace
    query = " ".join(query.split())
    # Remove common filler words (optional)
    # query = remove_stop_words(query)
    return query
```

📌 **Pro tip:** Monitor false positives (semantically similar but different intent) and false negatives (same intent but missed).

---

<details>
<summary><strong>🛒 Prompt Example — E-commerce Threshold Tuning (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to tune cache similarity threshold for e-commerce.
Return code only—no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Create test_threshold(threshold: float, test_queries: list) function
2. Test thresholds: [0.08, 0.10, 0.13, 0.15, 0.18, 0.20, 0.25]
3. Create test queries that match typical FAQ cache content:
   - Very close (distance ~0.05-0.10): "I want my money back", "Can I get a refund?"
   - Close (distance ~0.10-0.15): "What's your refund policy?", "Help me reset my password"
   - Moderate (distance ~0.15-0.20): "Where's my package?", "Stop my subscription"
   - Looser (distance ~0.20-0.25): "Update my shipping info", "What payment do you take?"
   - Should miss: "What are your business hours?", "Do you have a mobile app?"
4. Ensure cache has standard FAQ data (refunds, passwords, orders, shipping, subscriptions)
5. For each threshold, use cache.check(query, distance_threshold=threshold)
6. Calculate hit rate for EACH threshold separately (should increase as threshold increases)
7. If all thresholds show same hit rate, verify threshold is passed to cache.check()
8. Print threshold, hit rate, and sample matched queries
9. Create visualization showing hit rate increasing with threshold
10. Recommend optimal threshold based on F1 score

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>🌍 Prompt Example — Travel Threshold Tuning (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to tune cache threshold for travel recommendations.
Return code only—no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Create threshold_sweep(min: float, max: float, steps: int) function
2. Test with travel query pairs:
   - "Best beaches Europe" vs "Top European beach destinations"
   - "Affordable ski resorts" vs "Budget skiing locations"
3. Calculate precision and recall for each threshold
4. Plot precision-recall curve
5. Identify F1-optimal threshold
6. Test seasonal query variations

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>💻 Prompt Example — Code Documentation Threshold Tuning (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to tune threshold for code documentation cache.
Return code only—no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Create evaluate_threshold(threshold: float, code_queries: list) function
2. Test with programming question pairs:
   - "Sort array Python" vs "How to sort list Python"
   - "JavaScript async await" vs "JS async await syntax"
3. Measure hit rate and precision for different languages (Python, JavaScript, etc.)
4. Calculate precision, recall, and F1 score
5. Recommend optimal threshold based on results
6. Validate with edge cases

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

In [ ]:
# Empty cell for student to paste threshold tuning code

# Test & Evaluate Performance

Measure your semantic cache's effectiveness: hit rates, latency improvements, and cost savings.

- **Goal:** Quantify cache performance and validate it meets your spec goals.
- **Workflow:** Run realistic query workloads and measure key metrics.
- **Remember:** Real-world performance depends on query distribution-test with realistic patterns.

---

## Key Performance Metrics

**Cache Effectiveness:**
- **Hit Rate** = Hits / (Hits + Misses)
- **Precision** = Correct Hits / Total Hits (manual validation)
- **Coverage** = Unique queries in cache / Total unique queries

**Performance Gains:**
- **Latency Reduction** = (LLM_latency - Cache_latency) / LLM_latency
- **Cost Savings** = Hit_rate x LLM_cost_per_query
- **Throughput** = Queries per second

---

## Using the Evaluation Helpers

The `cache.evals` module provides `CacheEvaluator` and `PerfEval` classes:

### PerfEval - Track Latency and Costs

```python
from cache.evals import PerfEval

perf = PerfEval()
perf.set_total_queries(len(test_queries))

with perf:
    for query in test_queries:
        perf.start()
        results = cache.check(query)
        
        if results.matches:
            perf.tick("cache_hit")
            response = results.matches[0].response
        else:
            perf.tick("cache_miss")
            # Call LLM
            perf.start()
            response = llm(query)
            perf.tick("llm_call")
            perf.record_llm_call("gpt-4o-mini", query, response)
            cache.store(prompt=query, response=response)

# Get metrics
metrics = perf.get_metrics(labels=["cache_hit", "llm_call"])
costs = perf.get_costs()

print(perf.summary(labels=["cache_hit", "llm_call"]))
```

### CacheEvaluator - Measure Precision/Recall

```python
from cache.evals import CacheEvaluator

# After running queries, create evaluator with true labels
evaluator = CacheEvaluator(true_labels, cache_results)

# Get metrics at specific threshold
metrics = evaluator.get_metrics(distance_threshold=0.3)
print(f"Precision: {metrics['precision']:.2%}")
print(f"Recall: {metrics['recall']:.2%}")
print(f"F1 Score: {metrics['f1_score']:.2%}")

# Sweep thresholds to find optimal
sweep_results = evaluator.sweep_thresholds(metric_to_maximize="f1_score")
print(f"Best threshold: {sweep_results['best_threshold']:.3f}")
```

**Pro tip:** Test with both warm cache (queries repeated) and cold cache scenarios.

---

<details>
<summary><strong>Prompt Example - E-commerce Performance Testing (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to evaluate e-commerce cache performance.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Import PerfEval from cache.evals (use: from cache.evals import PerfEval)
2. Create test dataset with 50+ support queries (mix of similar and unique)
3. Use perf.start() before operations and perf.tick("label") after
4. Track "cache_hit" and "llm_call" labels
5. Create mock LLM function with realistic latency (~500ms)
6. Use perf.record_llm_call() to track costs
7. Print perf.summary(labels=["cache_hit", "llm_call"])
8. Calculate and print hit rate: cache_hits / total_queries
9. Compare against spec goals (65% hit rate target)

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>Prompt Example - Travel Cache Evaluation (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to evaluate travel recommendation cache.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Import from cache.evals (use: from cache.evals import PerfEval, CacheEvaluator)
2. Create diverse travel query dataset (beaches, skiing, cities, etc.)
3. Use perf.start() and perf.tick() pattern for timing
4. Test with labeled data to measure precision/recall
5. Use evaluator.sweep_thresholds() to find optimal threshold
6. Print perf.summary(labels=["cache_hit", "llm_call"])
7. Validate against 60%+ hit rate goal

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

<details>
<summary><strong>Prompt Example - Code Documentation Cache Testing (click to expand)</strong></summary>

```
You are my coding assistant. Generate Python code to test code documentation cache.
Return code only-no explanations, comments, or markdown.

Context: See attached spec.md

Requirements:
1. Import from cache.evals (use: from cache.evals import PerfEval, CacheEvaluator)
2. Create coding question dataset across languages (Python, JS, Java, etc.)
3. Use perf.start() before cache.check() and perf.tick("cache_hit") or perf.tick("llm_call")
4. Use perf.record_llm_call() for cost tracking
5. Test with common coding patterns (sorting, loops, async, etc.)
6. Calculate cost savings with perf.get_costs()
7. Print perf.summary(labels=["cache_hit", "llm_call"])

**Attachments:**
- `spec.md` for project info
- `docs.md` for documentation on coding with Redis
- `project.ipynb` to see the progress of my project
```

</details>

In [ ]:
# Empty cell for student to paste performance evaluation code

# 🎯 Summary & Next Steps

Congratulations! You've built a production-ready semantic caching system with Redis.

## What You've Accomplished

✅ Designed a semantic caching strategy tailored to your domain
✅ Configured Redis vector search with optimal parameters
✅ Implemented embedding-based semantic similarity
✅ Built core caching logic with hit/miss handling
✅ Tuned similarity thresholds for precision vs coverage
✅ Measured performance gains and cost savings

## Performance Targets Review

Compare your results against typical production benchmarks:

| Metric | Target | Typical Range |
|--------|--------|---------------|
| Hit Rate | 60-70% | 40-80% |
| Latency Reduction | 70-80% | 50-90% |
| Cost Savings | 70%+ | 50-85% |
| Precision | >95% | 85-99% |
| Cache Lookup | <5ms | 1-10ms |

## Next Steps

### Production Enhancements
1. **Advanced Techniques:**
   - Add fuzzy matching + semantic search (hybrid approach)
   - Implement cross-encoder reranking for higher precision
   - Use LLM-as-a-judge for cache quality validation

2. **Scale & Monitoring:**
   - Set up cache size limits and eviction policies
   - Add monitoring dashboards (hit rates, latency, errors)
   - Implement A/B testing for threshold optimization

3. **Integration:**
   - Integrate with LangChain/LangGraph agents
   - Add cache warming strategies
   - Implement distributed caching for high availability

### Learning Resources
- **Redis Vector Search Docs**: https://redis.io/docs/latest/develop/ai/
- **RedisVL Library**: https://docs.redisvl.com/
- **Semantic Caching Guide**: https://redis.io/blog/what-is-semantic-caching/
- **Course Materials**: Review the course materials from DeepLearning.AI

## Final Thoughts

Semantic caching is a powerful technique for:
- Reducing operational costs (70%+ savings possible)
- Improving user experience (80%+ faster responses)
- Scaling AI applications efficiently

The key is finding the right balance between **hit rate** (more cache hits) and **precision** (correct responses). Monitor both metrics continuously and adjust thresholds based on user feedback.

---

*Ready to deploy to production? Remember to set up proper monitoring and start with a conservative threshold!* 🚀